# vigilex – Phase I
## Notebook 03: Modellierung – Recall Risk Prediction

**Ziel:**
- Feature-Matrix aus Notebook 02 laden
- Baseline: Logistic Regression, Random Forest
- Hauptmodell: LightGBM + Optuna Hyperparameter-Tuning
- Evaluation: Recall-fokussiert (kein FN-Risiko bei Patientensicherheit)
- Feature Importance + SHAP-Analyse
- Modell speichern für Streamlit Demo

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report, roc_auc_score,
    precision_recall_curve, average_precision_score,
    ConfusionMatrixDisplay
)
import lightgbm as lgb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

DATA_PROC  = Path('../data/processed')
MODELS_DIR = Path('../models')
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print('Setup OK')

## 1. Daten laden & Split-Strategie

**Wichtig:** Zeitlicher Split, kein zufälliger Split – sonst Data Leakage!
- Train: Alles bis Ende 2022
- Validation: 2023
- Test: 2024–2025

In [ ]:
df = pd.read_parquet(DATA_PROC / 'features_labeled.parquet')
df['date_of_event'] = pd.to_datetime(df['date_of_event'])

FEATURE_COLS = [
    'complaints_7d', 'complaints_30d', 'complaints_90d',
    'severe_events_7d', 'severe_events_30d', 'severe_events_90d',
    'complaint_accel', 'max_severity', 'mean_severity',
]
TARGET = 'recall_label'

train_mask = df['date_of_event'] < '2023-01-01'
val_mask   = (df['date_of_event'] >= '2023-01-01') & (df['date_of_event'] < '2024-01-01')
test_mask  = df['date_of_event'] >= '2024-01-01'

X_train, y_train = df.loc[train_mask, FEATURE_COLS], df.loc[train_mask, TARGET]
X_val,   y_val   = df.loc[val_mask,   FEATURE_COLS], df.loc[val_mask,   TARGET]
X_test,  y_test  = df.loc[test_mask,  FEATURE_COLS], df.loc[test_mask,  TARGET]

print(f'Train:      {len(X_train):>6,} Rows | Recall Rate: {y_train.mean():.1%}')
print(f'Validation: {len(X_val):>6,} Rows | Recall Rate: {y_val.mean():.1%}')
print(f'Test:       {len(X_test):>6,} Rows | Recall Rate: {y_test.mean():.1%}')

## 2. Baseline Modelle

In [ ]:
def evaluate_model(model, X_val, y_val, model_name: str) -> dict:
    """Gibt ROC-AUC, PR-AUC und Classification Report aus."""
    y_prob = model.predict_proba(X_val)[:, 1]
    y_pred = model.predict(X_val)

    roc_auc = roc_auc_score(y_val, y_prob)
    pr_auc  = average_precision_score(y_val, y_prob)

    print(f'\n=== {model_name} ===')
    print(f'ROC-AUC: {roc_auc:.4f} | PR-AUC: {pr_auc:.4f}')
    print(classification_report(y_val, y_pred, target_names=['Kein Recall', 'Recall']))

    return {'model': model_name, 'roc_auc': roc_auc, 'pr_auc': pr_auc}


results = []

# Logistic Regression
lr = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42))
])
lr.fit(X_train, y_train)
results.append(evaluate_model(lr, X_val, y_val, 'Logistic Regression'))

# Random Forest
rf = RandomForestClassifier(
    n_estimators=200, class_weight='balanced',
    max_depth=8, random_state=42, n_jobs=-1
)
rf.fit(X_train, y_train)
results.append(evaluate_model(rf, X_val, y_val, 'Random Forest'))

## 3. LightGBM + Optuna Tuning

In [ ]:
# Class imbalance: positives gewichten
scale_pos = (y_train == 0).sum() / (y_train == 1).sum()
print(f'Scale Pos Weight: {scale_pos:.1f}')

def lgb_objective(trial):
    params = {
        'objective':        'binary',
        'metric':           'average_precision',
        'verbosity':        -1,
        'boosting_type':    'gbdt',
        'scale_pos_weight': scale_pos,
        'n_estimators':     trial.suggest_int('n_estimators', 100, 800),
        'learning_rate':    trial.suggest_float('learning_rate', 1e-3, 0.2, log=True),
        'num_leaves':       trial.suggest_int('num_leaves', 16, 128),
        'max_depth':        trial.suggest_int('max_depth', 3, 12),
        'min_child_samples':trial.suggest_int('min_child_samples', 10, 100),
        'subsample':        trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha':        trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        'reg_lambda':       trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
    }
    model = lgb.LGBMClassifier(**params, random_state=42)
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(50, verbose=False)]
    )
    y_prob = model.predict_proba(X_val)[:, 1]
    return average_precision_score(y_val, y_prob)


study = optuna.create_study(direction='maximize')
study.optimize(lgb_objective, n_trials=50, show_progress_bar=True)

print(f'\nBeste PR-AUC: {study.best_value:.4f}')
print(f'Beste Params: {study.best_params}')

In [ ]:
# Finales LightGBM Modell mit besten Params
best_params = study.best_params
best_params.update({'objective': 'binary', 'verbosity': -1,
                    'scale_pos_weight': scale_pos, 'random_state': 42})

lgbm = lgb.LGBMClassifier(**best_params)
lgbm.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(50, verbose=False)]
)

results.append(evaluate_model(lgbm, X_val, y_val, 'LightGBM (Optuna)'))

## 4. Modellvergleich & Threshold-Optimierung

Bei Patientensicherheits-Anwendungen: **hohen Recall priorisieren** (lieber False Positive als False Negative).

In [ ]:
# Precision-Recall Kurven
fig, ax = plt.subplots(figsize=(8, 6))

for result, model, color in zip(results, [lr, rf, lgbm], ['blue', 'green', 'red']):
    y_prob = model.predict_proba(X_val)[:, 1]
    prec, rec, _ = precision_recall_curve(y_val, y_prob)
    ax.plot(rec, prec, color=color,
            label=f'{result["model"]} (PR-AUC={result["pr_auc"]:.3f})')

ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Kurven')
ax.legend()
plt.tight_layout()
plt.savefig('../data/precision_recall_curves.png', dpi=150)
plt.show()

In [ ]:
# Optimalen Threshold für LightGBM bestimmen (F2-Score: Recall doppelt gewichtet)
from sklearn.metrics import fbeta_score

y_prob_val = lgbm.predict_proba(X_val)[:, 1]
thresholds = np.arange(0.1, 0.9, 0.02)
f2_scores  = [fbeta_score(y_val, (y_prob_val >= t).astype(int), beta=2) for t in thresholds]

best_thresh = thresholds[np.argmax(f2_scores)]
print(f'Optimaler Threshold (F2): {best_thresh:.2f}')
print(f'F2-Score dabei:           {max(f2_scores):.4f}')

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(thresholds, f2_scores)
ax.axvline(best_thresh, color='red', linestyle='--', label=f'Best: {best_thresh:.2f}')
ax.set_xlabel('Threshold')
ax.set_ylabel('F2-Score')
ax.set_title('Threshold-Optimierung (F2)')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Feature Importance

In [ ]:
importance = pd.Series(
    lgbm.feature_importances_,
    index=FEATURE_COLS
).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
importance.plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('LightGBM Feature Importance')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.savefig('../data/feature_importance.png', dpi=150)
plt.show()

## 6. Test-Set Evaluation & Modell speichern

In [ ]:
print('=== FINALE TEST-SET EVALUATION ===')
y_prob_test = lgbm.predict_proba(X_test)[:, 1]
y_pred_test = (y_prob_test >= best_thresh).astype(int)

print(f'ROC-AUC: {roc_auc_score(y_test, y_prob_test):.4f}')
print(f'PR-AUC:  {average_precision_score(y_test, y_prob_test):.4f}')
print(f'F2:      {fbeta_score(y_test, y_pred_test, beta=2):.4f}')
print()
print(classification_report(y_test, y_pred_test, target_names=['Kein Recall', 'Recall']))

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_test,
    display_labels=['Kein Recall', 'Recall'],
    ax=ax, colorbar=False
)
plt.title('Confusion Matrix (Test-Set)')
plt.tight_layout()
plt.savefig('../data/confusion_matrix.png', dpi=150)
plt.show()

In [ ]:
# Modell + Metadaten speichern
model_artifact = {
    'model':       lgbm,
    'threshold':   best_thresh,
    'feature_cols': FEATURE_COLS,
    'best_params': best_params,
}
joblib.dump(model_artifact, MODELS_DIR / 'lgbm_recall_risk.joblib')
print(f'Modell gespeichert: {MODELS_DIR / "lgbm_recall_risk.joblib"}')

## Zusammenfassung

| Modell | ROC-AUC | PR-AUC |
|---|---|---|
| Logistic Regression | – | – |
| Random Forest | – | – |
| **LightGBM (Optuna)** | **–** | **–** |

*(Werte werden beim Ausführen befüllt)*

→ Weiter mit **Notebook 04: RAG Agent** (Phase II)